GRM / Genotype Preparation
Prepare a high-quality, pruned genotype set for Step-1 ridge regression.

This ensures that only common, well-QC’d variants with consistent IDs are retained,
providing REGENIE with a clean, reliable input for building genome-wide polygenic predictors.

In [1]:
# Import required libraries
import sys
import os 
import numpy as np
import pandas as pd
from datetime import datetime

# Ensuring dsub is up to date
#!pip3 install --upgrade dsub

## Load in Variables

In [2]:
%run /home/jupyter/workspace/workspace-scripts/Var_set_2.ipynb

bucket_input:   gs://workspace-bucket-wb-blinding-cabbage-2295
bucket_output:  gs://workspace-output-wb-blinding-cabbage-2295
bucket_scripts: gs://script-wb-blinding-cabbage-2295
bucket_GenDat: gs://workspace-gendat-wb-blinding-cabbage-2295
dataset:        wb-silky-artichoke-2408.C2024Q3R8


In [3]:
# Save this Python variable as an environment variable so that its easier to use within %%bash cells
%env JOB_ID={LINE_COUNT_JOB_ID}
#my_bucket = os.environ['WORKSPACE_BUCKET']
pd.set_option('display.max_colwidth', 0)

results='grm' # Directory for results for dsu
%env results={results}

USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')
%env USER_NAME={USER_NAME}

JOB_NAME='clinvar_v8' # Job Name for dsu
%env JOB_NAME={JOB_NAME}

# ANCESTRY="EUR"
# %env ANCESTRY={ANCESTRY}

env: JOB_ID={LINE_COUNT_JOB_ID}
env: results=grm
env: USER_NAME='nathani
env: JOB_NAME=clinvar_v8


In [4]:
# Analysis Results Folder
output_folder = os.path.join(
    bucket_GenDat,
    results,
    JOB_NAME
)
%env output_folder={output_folder}

output_logs_folder = os.path.join(output_folder, "logs") # Log files folder
%env output_logs_folder={output_logs_folder}

env: output_folder=gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8
env: output_logs_folder=gs://workspace-gendat-wb-blinding-cabbage-2295/grm/clinvar_v8/logs


# **PLINK format**

## Pruning

### make a script for pruning, MAF, Geno, hwe and autosome filtering

In [5]:
%%writefile /home/jupyter/workspace/workspace-scripts/dsub_Plink_auto.sh

set -o pipefail 
set -o errexit

plink2 \
    --bfile "${bed_file%.bed}" \
    --chr 1-22 \
    --maf 0.01 \
    --geno 0.1 \
    --hwe 1e-6 \
    --indep-pairwise 50 5 0.2 \
    --threads 7 \
    --memory 30000 \
    --make-bed \
    --out "${output_folder}/autosomes_pruned"


echo "output_folder: ${output_folder}"


Overwriting /home/jupyter/workspace/workspace-scripts/dsub_Plink_auto.sh


In [11]:
!echo ${GOOGLE_PROJECT}

'wb-blinding-cabbage-2295'


In [26]:
%%bash --out filtering
#########################################
#Tutorial Script
#########################################
dsub \
    --provider google-batch \
    --user-project wb-blinding-cabbage-2295 \
    --project wb-blinding-cabbage-2295 \
    --network "global/networks/network" \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --use-private-address \
    --user "${DSUB_USER_NAME}" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --name "${JOB_NAME}" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/biocontainer/plink2:alpha2.3_jan2020" \
    --machine-type "n2-highmem-2" \
    --timeout "14d" \
    --boot-disk-size 20 \
    --output-recursive out_path="${output_folder}" \
    --command 'echo hello world > ${out_path}/test11.txt && \
             plink2 --help > ${out_path}/help_plink.txt && \
             nproc > ${out_path}/vm_cpu.txt && \
             cat /proc/meminfo | grep MemTotal > ${out_path}/vm_ram.txt && \
             df -h > ${out_path}/vm_disk.txt'

Job properties:
  job-id: clinvar-v8--jupyter--260629-053349-26
  job-name: clinvar-v8
  user-id: jupyter
Provider internal-id (operation): projects/wb-blinding-cabbage-2295/locations/us-central1/jobs/clinvar-v8--jupyter--260629-053349-26-0-0
Launched job-id: clinvar-v8--jupyter--260629-053349-26
To check the status, run:
  dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260629-053349-26' --users 'jupyter' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260629-053349-26' --users 'jupyter'


In [27]:
  !dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--jupyter--260629-053349-26' --users 'jupyter' --status '*'

Job Name    Status    Last Update
----------  --------  --------------
clinvar-v8  SUCCESS   06-29 05:35:46



In [44]:
!echo "${OWNER_EMAIL}" | cut -d@ -f1 | tr '.' '-' | tr -d '\r'

'nathani


In [45]:
%%bash --out filtering

# Get a shorter username to leave more characters for the job name
DSUB_USER_NAME=nathani


MACHINE_TYPE="n2-standard-8"
#MACHINE_TYPE="n2-standard-16"
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")
# Change for your bucket, path in output of cell directly above:
#BASH_SCRIPT="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_auto.sh"
BASH_SCRIPT="gs://script-wb-blinding-cabbage-2295/dsub_Plink_auto.sh"
#--image "gcr.io/bick-aps2/plink2:polygenic" \
#google-cls-v2
dsub \
    --provider google-batch \
    --user-project wb-blinding-cabbage-2295 \
    --project wb-blinding-cabbage-2295 \
    --network "global/networks/network" \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --use-private-address \
    --user "${DSUB_USER_NAME}" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/ubuntu:latest" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --name "${JOB_NAME}" \
    --image "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod/biocontainer/plink2:alpha2.3_jan2020" \
    --preemptible \
    --boot-disk-size 100 \
    --disk-size 200 \
    --timeout "2d" \
    --machine-type ${MACHINE_TYPE} \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=wb-blinding-cabbage-2295 \
    --env output_folder="${output_folder}" \
    --input bed_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bim_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input fam_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --output-recursive output_folder="${output_folder}"

httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Job properties:
  job-id: clinvar-v8--nathani--260629-062026-39
  job-name: clinvar-v8
  user-id: nathani
Provider internal-id (operation): projects/wb-blinding-cabbage-2295/locations/us-central1/jobs/clinvar-v8--nathani--260629-062026-39-0-0
Launched job-id: clinvar-v8--nathani--260629-062026-39
To check the status, run:
  dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260629-062026-39' --users 'nathani' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260629-062026-39' --users 'nathani'


In [48]:
!dstat --provider google-batch --project wb-blinding-cabbage-2295 --location us-central1 --jobs 'clinvar-v8--nathani--260629-062026-39' --users 'nathani' --status '*'

Job Name    Status    Last Update
----------  --------  --------------
clinvar-v8  RUNNING   06-29 06:21:53



In [29]:
%%bash --out filtering

# Get a shorter username to leave more characters for the job name
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

MACHINE_TYPE="n2-standard-8"
#MACHINE_TYPE="n2-standard-16"
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")
# Change for your bucket, path in output of cell directly above:
#BASH_SCRIPT="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_auto.sh"
BASH_SCRIPT="gs://script-wb-blinding-cabbage-2295/dsub_Plink_auto.sh"
#google-cls-v2
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "gcr.io/bick-aps2/plink2:polygenic" \
    --network "global/networks/network" \
    --use-private-address \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --boot-disk-size 100 \
    --disk-size 200 \
    --machine-type ${MACHINE_TYPE} \
    --name "${JOB_NAME}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --env output_folder="${output_folder}" \
    --input bed_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
    --input bim_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
    --input fam_file="gs://vwb-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
    --output-recursive output_folder="${output_folder}"

httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
Job properties:
  job-id: clinvar-v8---nathani--260629-060350-85
  job-name: clinvar-v8
  user-id: -nathani
Traceback (most recent call last):
  File "/opt/conda/envs/jupyter/lib/python3.10/site-packages/google/api_core/grpc_helpers.py", line 76, in error_remapped_callable
    return callable_(*args, **kwargs)
  File "/opt/conda/envs/jupyter/lib/python3.10/site-packages/grpc/_interceptor.py", line 277, in __call__
    response, ignored_call = self._with_call(
  File "/opt/conda/envs/jupyter/lib/python3.10/site-packages/grpc/_interceptor.py", line 332, in _with_call
    return call.result(), call
  File "/opt/conda/envs/jupyter/lib/python3.10/site-packages/grpc/_channel.py", line 439, in result
    raise self
  File "/opt/conda/envs/jupyter/lib/python3.

CalledProcessError: Command 'b'\n# Get a shorter username to leave more characters for the job name\nDSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"\n\nMACHINE_TYPE="n2-standard-8"\n#MACHINE_TYPE="n2-standard-16"\nTIMESTAMP=$(date +"%Y%m%d_%H%M%S")\n# Change for your bucket, path in output of cell directly above:\n#BASH_SCRIPT="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_auto.sh"\nBASH_SCRIPT="gs://script-wb-blinding-cabbage-2295/dsub_Plink_auto.sh"\n#google-cls-v2\n    dsub \\\n    --provider google-batch \\\n    --user-project "${GOOGLE_PROJECT}" \\\n    --project "${GOOGLE_PROJECT}" \\\n    --image "gcr.io/bick-aps2/plink2:polygenic" \\\n    --network "global/networks/network" \\\n    --use-private-address \\\n    --subnetwork "regions/us-central1/subnetworks/subnetwork" \\\n    --service-account "$(gcloud config get-value account)" \\\n    --user "${DSUB_USER_NAME}" \\\n    --regions us-central1 \\\n    --logging "${output_logs_folder}/${TIMESTAMP

In [5]:
#check job status
!dstat --provider google-batch --project terra-vpc-sc-d55470c6 --location us-central1 --jobs 'microarray--nathani--250908-032553-46' --users 'nathani' --status '*'

Job Name    Status                                      Last Update
----------  ------------------------------------------  --------------
microarray  Job state is set from RUNNING to SUCCEE...  09-08 06:43:42



## extract pruned SNPs

In [48]:
%%writefile ~/dsub_Plink_extract.sh

set -o pipefail 
set -o errexit


plink2 \
    --bfile "${bed_file%.bed}" \
    --extract "${in_file}" \
    --threads 3 \
    --memory 12000 \
    --make-bed \
    --out "${output_folder}/autosomes_final_pruned"


Overwriting /home/jupyter/dsub_Plink_extract.sh


In [58]:
%%bash --out extracting

# Get a shorter username to leave more characters for the job name
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

MACHINE_TYPE="n2-standard-4"
#MACHINE_TYPE="n2-standard-16"
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")
# Change for your bucket, path in output of cell directly above:
BASH_SCRIPT="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/scripts/dsub_Plink_extract.sh"

#clear log folder
gsutil -m rm "${output_logs_folder}/**"

#google-cls-v2
    dsub \
    --provider google-batch \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --image "gcr.io/bick-aps2/plink2:polygenic" \
    --network "global/networks/network" \
    --use-private-address \
    --subnetwork "regions/us-central1/subnetworks/subnetwork" \
    --service-account "$(gcloud config get-value account)" \
    --user "${DSUB_USER_NAME}" \
    --regions us-central1 \
    --logging "${output_logs_folder}/${TIMESTAMP}-{job-id}-{task-id}.log" \
    "$@" \
    --preemptible \
    --boot-disk-size 100 \
    --disk-size 400 \
    --machine-type ${MACHINE_TYPE} \
    --name "${JOB_NAME}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --env output_folder="${output_folder}" \
    --input bed_file="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bed" \
    --input bim_file="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bim" \
    --input fam_file="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.fam" \
    --input in_file="gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.in" \
    --output-recursive output_folder="${output_folder}"

Removing gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_034037-microarray--nathani--250909-034040-27-task-stderr.log...
Removing gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_034037-microarray--nathani--250909-034040-27-task-stdout.log...
Removing gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_034037-microarray--nathani--250909-034040-27-task.log...
/ [3/3 objects] 100% Done                                                       
Operation completed over 3 objects.                                              
Job properties:
  job-id: microarray--nathani--250909-035021-81
  job-name: microarray
  user-id: nathani
Provider internal-id (operation): projects/terra-vpc-sc-d55470c6/locations/us-central1/jobs/microarray--nathani--250909-035021-81-0-0
Launched job-id: microarray--nathani--250909-035021-81
To check the status, run:
  dstat --provider google-batch --project t

In [59]:
%env JOB_ID={extracting}

env: JOB_ID=microarray--nathani--250909-035021-81


In [62]:
%%bash

dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

Job Name    Status                                      Last Update
----------  ------------------------------------------  --------------
microarray  Job state is set from RUNNING to SUCCEE...  09-09 03:57:56



In [63]:
!gsutil ls {output_logs_folder}

gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_035018-microarray--nathani--250909-035021-81-task-stderr.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_035018-microarray--nathani--250909-035021-81-task-stdout.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/logs/20250909_035018-microarray--nathani--250909-035021-81-task.log


In [68]:
!gsutil ls {output_folder}

gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.bed
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.bim
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.fam
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bed
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.bim
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.fam
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.log
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.in
gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_pruned.prune.out
gs://fc-secure-0c09c643-6

In [69]:
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.bim | wc -l
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.fam | wc -l

74906
447278


In [71]:
!gsutil cat gs://fc-secure-0c09c643-6594-4ba0-8f4b-29b4d72925ae/grm/microarray/PLINK/autosomes_final_pruned.fam | cut -f1 | sort | uniq -c

 447278 0


In [ ]:
!gsutil -{PROJECT_ID} 